In [1]:
1+1

2

In [2]:
import os
%pwd

'f:\\MS\\text-summarization-transformers\\research'

In [3]:
os.chdir("../")
%pwd

'f:\\MS\\text-summarization-transformers'

### Basic Configuration

In [16]:
from dataclasses import dataclass
from pathlib import Path

@dataclass
class DataIngestionConfig:
    root_dir: Path
    source_URL: Path
    local_data_file: Path
    unzip_dir: Path

In [17]:
from src.text_summarization_transformers.constants import *
from src.text_summarization_transformers.utils.common import read_yaml, create_directories


## Configuration Upadates

In [18]:
class ConfiguartionManager:
    def __init__(self, config_filepath=CONFIG_FILE_PATH, params_filepath=PARAMS_FILE_PATH):
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        create_directories([self.config.artifacts_root])

    def get_data_ingestion_config(self) -> DataIngestionConfig:
        config = self.config.data_ingestion
        create_directories([config.root_dir])

        data_ingestion_config=DataIngestionConfig(
            root_dir=Path(config.root_dir),
            source_URL=config.source_URL,
            local_data_file=Path(config.local_data_file),
            unzip_dir=Path(config.unzip_dir)

        )
        return data_ingestion_config

In [19]:
import os
import urllib.request as request
import zipfile
from src.text_summarization_transformers.logging import logger

## Components

In [20]:
class DataIngestion:
    def __init__(self, config: DataIngestionConfig):
        self.config = config

    def download_file(self):
        if not os.path.exists(self.config.local_data_file):
            filename, headers = request.urlretrieve(
                url=self.config.source_URL,
                filename=self.config.local_data_file
            )
            logger.info(f"{filename} downloaded with following info: \n{headers}")
        else:
            logger.info(f"File already exists of size: {round(os.path.getsize(self.config.local_data_file)/1024/1024, 2)} MB")

    def extract_zip_file(self):
        """ 
        zip_file_path: str
        Extracts the zip file into the data directory
        function returns None
        """
        unzip_path = self.config.unzip_dir
        os.makedirs(unzip_path, exist_ok=True)
        with zipfile.ZipFile(self.config.local_data_file, 'r') as zip_ref:
            zip_ref.extractall(unzip_path)
        logger.info(f"File extracted at: {unzip_path}")

In [22]:
config = ConfiguartionManager()
data_ingestion_config = config.get_data_ingestion_config()
data_ingestion = DataIngestion(config=data_ingestion_config)

data_ingestion.download_file()
data_ingestion.extract_zip_file()

[2026-05-29 01:02:07,859:INFO:yaml file: config\config.yaml loaded successfully]
[2026-05-29 01:02:07,860:INFO:yaml file: params.yaml loaded successfully]
[2026-05-29 01:02:07,864:INFO:created directory at: artifacts]
[2026-05-29 01:02:07,868:INFO:created directory at: artifacts/data_ingestion]
[2026-05-29 01:02:07,951:INFO:File already exists of size: 7.54 MB]
[2026-05-29 01:02:08,100:INFO:File extracted at: artifacts\data_ingestion]
